In [1]:
import sys
if "viashap_paper/src" not in sys.path:
    sys.path.append("viashap_paper/src")

import torch
from torch.utils.data import DataLoader

from models.mlp_models import MLPShapleyNetwork
from via_shap.via_shap import ViaShapModel
from samplers.uniform_sampler import UniformFeatureSampler
from loss_functions.shapley_regression_loss import ShapleyRegressionLoss
from loss_functions.prediction_loss import PredictionLoss
from loss_functions.value_functions import baseline_removal_value_fn

In [15]:
shapley_net = MLPShapleyNetwork(n_features=10, d_out=1, hidden_dims=[64, 64])
model = ViaShapModel(shapley_net)

model.eval()

sampler = UniformFeatureSampler()
sampler.set_seed(101)

shapley_loss = ShapleyRegressionLoss(
    value_fn=baseline_removal_value_fn,
    sampler=sampler,
    beta=10.0
)

pred_loss = PredictionLoss(torch.nn.MSELoss())

torch.manual_seed(101)
# x = torch.randn(32, 10)
x = torch.ones(32, 10)
y = torch.randn(32, 1)

y_pred = model.predict(x)

loss_shapley = shapley_loss(model, x, y)
loss_pred = pred_loss(y_pred, y)

total_loss = loss_shapley + loss_pred

total_loss.backward()


In [17]:
batch_size = x.shape[0]
d_out = y.shape[-1]

masked_x, masks = sampler.sample(x, n_coalitions=batch_size)
n_coalitions = masked_x.shape[0] // batch_size

value_est = baseline_removal_value_fn(model, masked_x)
shapley_values = model.shapley_network(x)

masks = masks.view(batch_size, n_coalitions, -1)

shapley_values_exp = shapley_values.unsqueeze(1).expand(-1, n_coalitions, -1, -1)
shapley_sum = torch.einsum("bkn, bknf -> bkf", masks, shapley_values_exp)

pred_baseline = model.predict(torch.zeros_like(x))  # (batch_size, d_out)

value_est = value_est.view(batch_size, n_coalitions, d_out)

loss_shapley = (
    (value_est - pred_baseline.unsqueeze(1) - shapley_sum).pow(2)
).mean()


In [18]:
loss_shapley.item()

0.04106944799423218

In [8]:
value_est.shape

torch.Size([32, 32, 1])

In [9]:
shapley_values.shape

torch.Size([32, 10, 1])

# With trainable bias

In [23]:
shapley_net = MLPShapleyNetwork(n_features=10, d_out=1, hidden_dims=[64, 64])
model = ViaShapModel(shapley_net, add_trainable_bias=True)

model.eval()

sampler = UniformFeatureSampler()
sampler.set_seed(101)

shapley_loss = ShapleyRegressionLoss(
    value_fn=baseline_removal_value_fn,
    sampler=sampler,
    beta=10.0å
)

# pred_loss = PredictionLoss(torch.nn.MSELoss())

torch.manual_seed(101)
# x = torch.randn(32, 10)
x = torch.ones(32, 10)
y = torch.randn(32, 1)

# y_pred = model.predict(x)

loss_shapley = shapley_loss(model, x, y)

In [24]:
loss_shapley.item()

0.7470139265060425